# Hugging Face へ「マージ済み（full）」SFTモデルをアップロードするノートブック

このノートブックは、あなたの `runs/sft/.../checkpoints/final/` にある **マージ済みモデル一式**（LoRAではない）を  
Hugging Face Hub の **Model Repository** にアップロードするための“専用アップローダ”です。

- ✅ `huggingface_hub.HfApi().upload_folder()` を使う方式（添付ノートブックのアップロード方式を踏襲）
- ✅ README.md（モデルカード）を作成してからアップロード（READMEが無い場合はエラー）
- ✅ アップロード対象をホワイトリストで制御（余計なログや一時ファイルを除外）

> 事前条件: Hugging Face の **WRITE権限トークン**が必要です。


## Step 0: 依存関係（必要なら）

すでに `huggingface_hub` が入っていればスキップしてOKです。

In [1]:
# 必要ならインストール
# !pip install -U "huggingface_hub>=0.23.0"

import huggingface_hub
print("huggingface_hub:", huggingface_hub.__version__)


huggingface_hub: 0.36.0


## Step 1: Hugging Face へログイン

- 推奨: 環境変数 `HF_TOKEN` にトークンを入れておく（シェルで export してから起動）
- ない場合は `login()` で対話入力します

In [2]:
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN", "").strip()
if token:
    login(token=token)
    print("✅ Logged in with HF_TOKEN (env).")
else:
    # 対話入力（Notebook で入力プロンプトが出ます）
    login()
    print("✅ Logged in interactively.")


✅ Logged in interactively.


## Step 2: 設定（アップロード元フォルダ / リポジトリID / 公開設定など）

ここだけあなたの環境に合わせて指定します。

In [3]:
import os
from pathlib import Path

# --- あなたのアップロード元（マージ済みモデルのフォルダ） ---
MODEL_DIR = Path(os.environ.get(
    "MODEL_DIR",
    "BEST/runs/sft/20260222_171949_sft_14e7a39/checkpoints/final"
)).expanduser().resolve()

# --- アップロード先 (例: 'your/qwen3-4b-sft-structured') ---
HF_REPO_ID = os.environ.get("HF_REPO_ID", "n4/Qwen3-4B-Instruct-2507-sft_166")

# --- 非公開にするなら True（環境変数 '1'/'true' で True） ---
PRIVATE = os.environ.get("HF_PRIVATE", "0").lower() in ("1","true","yes","y")

# --- アップロード時のコミットメッセージ ---
COMMIT_MESSAGE = os.environ.get("HF_COMMIT_MESSAGE", "Upload merged SFT model")

# --- ステージング（一時コピー）先 ---
STAGE_DIR = Path(os.environ.get("HF_STAGE_DIR", "./hf_upload_stage")).expanduser().resolve()

print("MODEL_DIR:", MODEL_DIR)
print("HF_REPO_ID:", HF_REPO_ID)
print("PRIVATE:", PRIVATE)
print("STAGE_DIR:", STAGE_DIR)


MODEL_DIR: /home/ubuntu/data_ubuntu/LLM2025/Advanced_Day8/BEST/runs/sft/20260222_171949_sft_14e7a39/checkpoints/final
HF_REPO_ID: n4/Qwen3-4B-Instruct-2507-sft_166
PRIVATE: True
STAGE_DIR: /home/ubuntu/data_ubuntu/LLM2025/Advanced_Day8/hf_upload_stage


## Step 3: README.md（モデルカード）を作る

Hugging Face では `README.md` がモデルカードです。最低限ここを埋めてからアップロードしてください。

このセルは **テンプレ生成**なので、生成後に README.md を直接編集してもOKです。

In [4]:
from pathlib import Path
import json
import textwrap

MODEL_DIR = Path(MODEL_DIR)

# config.json があれば読み、base_model 等の手掛かりにする（無くてもOK）
cfg = {}
cfg_path = MODEL_DIR / "config.json"
if cfg_path.exists():
    try:
        cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    except Exception:
        cfg = {}

base_model_guess = cfg.get("_name_or_path", "") or "Qwen/Qwen3-4B-Instruct-2507"

readme_path = MODEL_DIR / "README.md"
if not readme_path.exists():
    template = textwrap.dedent(f"""    ---
    base_model: {base_model_guess}
    language:
    - ja
    license: apache-2.0
    library_name: transformers
    pipeline_tag: text-generation
    tags:
    - sft
    - qwen
    - merged
    ---

    # Qwen3-4B-Instruct-2507-sft_166

    This repository contains a **merged (full) SFT model** (not a LoRA adapter).

    ## Base model
    - `{base_model_guess}`

    ## What’s inside
    - Full model weights (`model-*.safetensors` shards)
    - Tokenizer files
    - Generation / chat template files (if present)

    ## How to use

    ```python
    from transformers import AutoTokenizer, AutoModelForCausalLM

    repo_id = "{HF_REPO_ID}"
    tok = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(repo_id, device_map="auto", trust_remote_code=True)

    prompt = "Please output the following information in JSON format: Name=naisy, Age=714"
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=512)
    print(tok.decode(out[0], skip_special_tokens=True))
    ```

    ## Training
    - Method: SFT
    - Notes: This model is uploaded as merged weights because it is **not** guaranteed to be compatible as an adapter with the base model.

    ## Limitations
    - Add your known limitations, safety notes, evaluation results, etc.

    ## License
    - Inherit the base model license + your additions.
    """)
    readme_path.write_text(template, encoding="utf-8")
    print("README.md template created:", readme_path)
else:
    print("README.md already exists:", readme_path)

print("\n--- README.md path ---")
print(readme_path)


README.md template created: /home/ubuntu/data_ubuntu/LLM2025/Advanced_Day8/BEST/runs/sft/20260222_171949_sft_14e7a39/checkpoints/final/README.md

--- README.md path ---
/home/ubuntu/data_ubuntu/LLM2025/Advanced_Day8/BEST/runs/sft/20260222_171949_sft_14e7a39/checkpoints/final/README.md


## Step 4: 必須ファイルの確認

`final/` には通常 `config.json` と `model.safetensors.index.json` と `model-*.safetensors` が入ります。

In [5]:
from pathlib import Path

MODEL_DIR = Path(MODEL_DIR)

required_files = {
    "README.md",
    "config.json",
    "model.safetensors.index.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
}

present = {p.name for p in MODEL_DIR.iterdir() if p.is_file()}
missing = [f for f in required_files if f not in present]

has_shard = any(p.name.startswith("model-") and p.suffix == ".safetensors" for p in MODEL_DIR.iterdir() if p.is_file())
if not has_shard:
    missing.append("model-*.safetensors (shards)")

if missing:
    raise RuntimeError(
        "アップロードを中止しました。必須ファイルが不足しています:\n"
        + "\n".join(f"- {m}" for m in missing)
        + "\n\nMODEL_DIR を確認し、必要なら README.md も作成してください。"
    )

print("必須ファイルの確認が完了しました。")


必須ファイルの確認が完了しました。


## Step 5: アップロード対象の選別（ホワイトリスト）とステージング

巨大なログ等を巻き込まないため、アップロード対象を許可パターンで制御します。

In [6]:
import fnmatch
import shutil
from pathlib import Path

MODEL_DIR = Path(MODEL_DIR)
STAGE_DIR = Path(STAGE_DIR)

ALLOW_PATTERNS = [
    "README.md",
    "config.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "model-*.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "added_tokens.json",
    "merges.txt",
    "vocab.json",
    "chat_template.jinja",
    "*.json",
    "*.jinja",
    "*.txt",
]

def is_allowed(name: str) -> bool:
    return any(fnmatch.fnmatch(name, pat) for pat in ALLOW_PATTERNS)

if STAGE_DIR.exists():
    shutil.rmtree(STAGE_DIR)
STAGE_DIR.mkdir(parents=True)

copied = []
skipped = []
for p in MODEL_DIR.iterdir():
    if p.is_file():
        if is_allowed(p.name):
            (STAGE_DIR / p.name).write_bytes(p.read_bytes())
            copied.append(p.name)
        else:
            skipped.append(p.name)

print("Copied to stage:", len(copied))
print("Skipped:", len(skipped))
print("Example copied:", sorted(copied)[:20])


Copied to stage: 13
Skipped: 0
Example copied: ['README.md', 'added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## Step 6: Hugging Face Hub にリポジトリ作成＆アップロード

In [7]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True,
    private=PRIVATE,
)

api.upload_folder(
    folder_path=str(STAGE_DIR),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message=COMMIT_MESSAGE,
)

print("アップロードが正常に完了しました。")
print(f"URL: https://huggingface.co/{HF_REPO_ID}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

## （任意）Step 7: ローカルでロード確認

アップロード前にローカル `MODEL_DIR` が `transformers` で読めるか確認したい場合に使います。

In [ ]:
# 注意: GPU/VRAMや環境によっては重いです。必要なければスキップしてください。
# from transformers import AutoTokenizer, AutoModelForCausalLM
# tok = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
# model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, device_map="auto", trust_remote_code=True)
# print("✅ loaded OK:", type(model))
